# Lead-Lag Networks in Metal Trading

Pipeline complet pour detecter les relations d'anticipation entre 8 actifs
(metaux, FX, taux) via Dynamic Time Warping et un graphe de transfert d'entropie.

Ce notebook est auto-suffisant : toutes les classes sont definies en ligne,
aucune dependance au dossier `src/`. Compatible Google Colab.

**Pipeline**

1. DataLoader : telechargement Yahoo Finance, diagnostic et imputation des manquants
2. Preprocessing : log-returns, filtres (Kalman, Butterworth, Savitzky-Golay, MA, EMA), regimes Markov
3. EDA : correlations multi-lag, clustermap DTW
4. LeadLagDTW : matrices de distance et de lead-lag entre toutes les paires
5. Tuning et Validation : Optuna + TimeSeriesSplit pour la bande Sakoe-Chiba
6. EntropyTransferGraph : graphe de transfert via JSD et MDS
7. Synthese : lectures finales et implications trading


## Setup

Installation des dependances (a decommenter sur Colab) et imports.
Le dossier `outputs/` recoit les figures et CSV intermediaires.

In [ ]:
# Decommenter sur Google Colab pour installer les dependances
# !pip install -q yfinance missingno statsmodels pykalman dtaidistance \
#                 optuna seaborn networkx scikit-learn scipy

In [ ]:
from __future__ import annotations

import os
import warnings
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

OUT = "outputs"
os.makedirs(OUT, exist_ok=True)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")

## Configuration et helpers partages

Liste des tickers (Bund10Y est proxie par l'ETF `IS0L.DE`, le rendement
`^TNX` n'ayant pas d'equivalent direct sur Yahoo).

Deux helpers reutilises plusieurs fois sont factorises ici :
`zscore` pour standardiser une serie, `dtw_distance_matrix` pour produire
la matrice de distance DTW entre toutes les paires d'un DataFrame.

In [ ]:
TICKERS = {
    "Gold": "GC=F",
    "Silver": "SI=F",
    "Oil": "CL=F",
    "EURUSD": "EURUSD=X",
    "JPYUSD": "JPYUSD=X",
    "DXY": "DX-Y.NYB",
    "UST10Y": "^TNX",
    "Bund10Y": "IS0L.DE",
}


def zscore(s) -> np.ndarray:
    a = np.asarray(s, dtype=float)
    return (a - a.mean()) / (a.std() + 1e-12)


def dtw_distance_matrix(df: pd.DataFrame, window: Optional[int] = None) -> pd.DataFrame:
    """Matrice n x n des distances DTW entre colonnes z-scorees."""
    from dtaidistance import dtw

    z = df.apply(zscore, axis=0)
    cols = list(df.columns)
    n = len(cols)
    D = np.zeros((n, n))
    kwargs = {"window": int(window)} if window is not None else {}
    for i in range(n):
        for j in range(i + 1, n):
            d = dtw.distance(z[cols[i]].values, z[cols[j]].values, **kwargs)
            D[i, j] = D[j, i] = d
    return pd.DataFrame(D, index=cols, columns=cols)

## Task 1 - DataLoader

Telechargement des 8 series sur 2020-2025 en quotidien, fusion par jointure
externe, diagnostic des manquants (test ANOVA-by-pattern, proxy de Little MCAR),
puis imputation forward-fill (les prix sont des stocks : la derniere valeur
observee reste valide jusqu'au prochain trade).

In [ ]:
@dataclass
class DataLoader:
    tickers: Dict[str, str]
    start_date: str
    end_date: str
    freq: str = "1d"
    price_col: str = "Close"
    raw: Dict[str, pd.Series] = field(default_factory=dict, init=False)
    merged: Optional[pd.DataFrame] = field(default=None, init=False)

    def fetch_data(self) -> Dict[str, pd.Series]:
        import yfinance as yf

        out: Dict[str, pd.Series] = {}
        for name, sym in self.tickers.items():
            try:
                df = yf.download(
                    sym, start=self.start_date, end=self.end_date,
                    interval=self.freq, progress=False, auto_adjust=False,
                )
                if df is None or df.empty:
                    print(f"[WARN] empty data for {name} ({sym})")
                    continue
                if isinstance(df.columns, pd.MultiIndex):
                    df.columns = df.columns.get_level_values(0)
                s = df[self.price_col].rename(name)
                if s.index.tz is not None:
                    s.index = s.index.tz_convert("UTC").tz_localize(None)
                out[name] = s
            except Exception as e:
                print(f"[ERROR] {name} ({sym}): {e}")
        self.raw = out
        return out

    def merge_data(self) -> pd.DataFrame:
        if not self.raw:
            raise ValueError("Aucune donnee a fusionner, appeler fetch_data() d'abord.")
        merged = pd.concat(self.raw.values(), axis=1, join="outer").sort_index()
        merged.columns = list(self.raw.keys())
        self.merged = merged
        return merged

    def eda_missing(self, df: Optional[pd.DataFrame] = None):
        import missingno as msno

        df = df if df is not None else self.merged
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        msno.matrix(df, ax=axes[0], sparkline=False)
        axes[0].set_title("Matrice des manquants")
        msno.heatmap(df, ax=axes[1])
        axes[1].set_title("Correlation des manquants")
        plt.tight_layout()
        return fig

    def missing_test(self, df: Optional[pd.DataFrame] = None) -> dict:
        """Proxy de Little MCAR : ANOVA par pattern de manquance."""
        from scipy import stats

        df = df if df is not None else self.merged
        patterns = df.isna().apply(lambda r: tuple(r.values), axis=1)
        results = {}
        for col in df.columns:
            obs = df[col].dropna()
            labels = patterns.loc[obs.index]
            groups = [obs.loc[labels == p].values for p in labels.unique()]
            groups = [g for g in groups if len(g) > 1]
            if len(groups) < 2:
                results[col] = {"F": np.nan, "p": np.nan}
                continue
            F, p = stats.f_oneway(*groups)
            results[col] = {"F": float(F), "p": float(p)}
        ps = [r["p"] for r in results.values() if not np.isnan(r["p"])]
        return {"per_column": results, "mcar_plausible": bool(ps) and all(p > 0.05 for p in ps)}

    def impute_data(self, df: Optional[pd.DataFrame] = None) -> pd.DataFrame:
        df = df if df is not None else self.merged
        out = df.ffill().bfill()
        if out.isna().any().any():
            raise RuntimeError("NaN residuels apres ffill/bfill.")
        self.merged = out
        return out

In [ ]:
dl = DataLoader(TICKERS, start_date="2020-01-01", end_date="2025-01-01", freq="1d")
dl.fetch_data()
merged = dl.merge_data()
print(f"Shape fusionnee : {merged.shape}")
print(f"NaN par colonne :\n{merged.isna().sum()}")

In [ ]:
fig = dl.eda_missing(merged)
fig.savefig(os.path.join(OUT, "01_missingness.png"), dpi=120, bbox_inches="tight")
plt.show()

mcar = dl.missing_test(merged)
print(f"MCAR plausible (p > 0.05 partout) : {mcar['mcar_plausible']}")

prices = dl.impute_data(merged)
prices.to_csv(os.path.join(OUT, "01_prices.csv"))
print(f"Dataset propre : {prices.shape}, NaN restants = {prices.isna().sum().sum()}")

## Task 2 - Preprocessing

Log-returns, RobustScaler, moyenne mobile. Puis 5 filtres compares sur
l'actif cible :

- Kalman local-level : smoother optimal sous random-walk gaussien
- Butterworth low-pass : passe-bande plate, retire le HF sans ripple
- Savitzky-Golay : fit polynomial local, preserve les pics
- Moving Average : baseline
- EMA (TA-Lib si dispo, sinon EWM pandas) : causal, utilisable en live

Detection de regimes via Markov-Switching (calme vs stress).

In [ ]:
@dataclass
class Preprocessing:
    data: pd.DataFrame
    scaled: Optional[pd.DataFrame] = field(default=None, init=False)
    log_returns: Optional[pd.DataFrame] = field(default=None, init=False)

    def transform_data(self, ma_window: int = 20) -> dict:
        from sklearn.preprocessing import RobustScaler

        prices = self.data.astype(float)
        log_ret = np.log(prices / prices.shift(1)).dropna()
        ma = prices.rolling(ma_window, min_periods=1).mean()
        scaled = pd.DataFrame(
            RobustScaler().fit_transform(prices.values),
            index=prices.index, columns=prices.columns,
        )
        self.log_returns, self.scaled = log_ret, scaled
        return {"log_returns": log_ret, "moving_average": ma, "scaled": scaled}

    def apply_kalman_filter(self, col: str) -> pd.Series:
        from pykalman import KalmanFilter

        s = self.data[col].dropna().astype(float)
        kf = KalmanFilter(
            transition_matrices=[1], observation_matrices=[1],
            initial_state_mean=s.iloc[0], initial_state_covariance=1.0,
            observation_covariance=1.0, transition_covariance=0.01,
        )
        means, _ = kf.smooth(s.values)
        return pd.Series(means.ravel(), index=s.index, name=f"{col}_kalman")

    def apply_butterworth_filter(self, col: str, cutoff: float = 0.1, order: int = 3) -> pd.Series:
        from scipy.signal import butter, filtfilt

        s = self.data[col].dropna().astype(float)
        b, a = butter(order, cutoff, btype="low")
        return pd.Series(filtfilt(b, a, s.values), index=s.index, name=f"{col}_butter")

    def apply_savgol_filter(self, col: str, window: int = 21, poly: int = 3) -> pd.Series:
        from scipy.signal import savgol_filter

        s = self.data[col].dropna().astype(float)
        if window % 2 == 0:
            window += 1
        return pd.Series(savgol_filter(s.values, window, poly), index=s.index, name=f"{col}_savgol")

    def apply_moving_average(self, col: str, window: int = 20) -> pd.Series:
        return self.data[col].rolling(window, min_periods=1).mean().rename(f"{col}_ma")

    def apply_ema(self, col: str, window: int = 20) -> pd.Series:
        s = self.data[col].dropna().astype(float)
        try:
            import talib
            return pd.Series(talib.EMA(s.values, timeperiod=window), index=s.index, name=f"{col}_ema")
        except ImportError:
            return s.ewm(span=window, adjust=False).mean().rename(f"{col}_ema")

    def apply_all_filters(self, col: str) -> pd.DataFrame:
        return pd.concat([
            self.data[col].rename(f"{col}_raw"),
            self.apply_kalman_filter(col),
            self.apply_butterworth_filter(col),
            self.apply_savgol_filter(col),
            self.apply_moving_average(col),
            self.apply_ema(col),
        ], axis=1)

    def detect_regimes_markov(self, col: str, k_regimes: int = 2) -> pd.Series:
        from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression

        ret = np.log(self.data[col]).diff().dropna()
        res = MarkovRegression(ret, k_regimes=k_regimes, trend="c",
                                switching_variance=True).fit(disp=False)
        return res.smoothed_marginal_probabilities.idxmax(axis=1).rename(f"{col}_regime")

In [ ]:
pp = Preprocessing(prices)
feats = pp.transform_data(ma_window=20)
print(f"Shape log-returns : {feats['log_returns'].shape}")

target = "Gold" if "Gold" in prices.columns else prices.columns[0]
filt = pp.apply_all_filters(target)
ax = filt.plot(figsize=(12, 5), title=f"{target} - comparaison des 5 filtres")
ax.figure.savefig(os.path.join(OUT, "02_filters.png"), dpi=120, bbox_inches="tight")
plt.show()

try:
    regimes = pp.detect_regimes_markov(target)
    print(f"Regimes Markov sur {target} :\n{regimes.value_counts()}")
except Exception as e:
    print(f"[skip] Regimes Markov : {e}")

## Task 3 - EDA

Correlations Pearson sur log-returns + correlations croisees a plusieurs
lags pour chaque paire. Clustermap DTW (z-score par serie pour comparer
des formes, pas des niveaux). Permet de detecter visuellement les
clusters d'actifs co-mouvants.

In [ ]:
@dataclass
class EDA:
    data: pd.DataFrame

    def correlation_matrix(self, max_lag: int = 5) -> dict:
        ret = np.log(self.data).diff().dropna()
        corr = ret.corr()
        cross = {}
        cols = ret.columns
        for i, a in enumerate(cols):
            for b in cols[i + 1:]:
                cross[(a, b)] = [ret[a].corr(ret[b].shift(lag)) for lag in range(1, max_lag + 1)]
        return {"corr": corr, "cross_corr": cross}

    def dtw_clustermap(self):
        import seaborn as sns

        ret = np.log(self.data).diff().dropna()
        D = dtw_distance_matrix(ret)
        g = sns.clustermap(D, cmap="viridis", annot=True, fmt=".1f")
        return g, D

In [ ]:
eda = EDA(prices)
corr = eda.correlation_matrix(max_lag=5)
print("Matrice de correlation (log-returns) :")
print(corr["corr"].round(2))
corr["corr"].to_csv(os.path.join(OUT, "03_corr.csv"))

In [ ]:
g, D_eda = eda.dtw_clustermap()
g.fig.savefig(os.path.join(OUT, "03_dtw_clustermap.png"), dpi=120, bbox_inches="tight")
plt.show()
D_eda.to_csv(os.path.join(OUT, "03_dtw_distance.csv"))

## Task 4 - LeadLagDTW

Pour chaque paire (X, Y), on calcule la distance DTW et le chemin de
warping optimal. Le lag signe est la moyenne de `(j - i)` sur le chemin :

- lag > 0 : X mene Y de `lag` pas
- lag < 0 : Y mene X

La bande Sakoe-Chiba (`window`) limite `|i - j| <= radius` et accelere
le calcul tout en evitant des alignements degeneres.

In [ ]:
@dataclass
class LeadLagDTW:
    data: pd.DataFrame
    sakoe_chiba_radius: Optional[int] = None
    distance_matrix_: Optional[pd.DataFrame] = field(default=None, init=False)
    lag_matrix_: Optional[pd.DataFrame] = field(default=None, init=False)

    def compute_dtw(self, s1, s2) -> Tuple[float, list]:
        from dtaidistance import dtw

        a, b = zscore(s1), zscore(s2)
        kwargs = {"window": int(self.sakoe_chiba_radius)} if self.sakoe_chiba_radius else {}
        return float(dtw.distance(a, b, **kwargs)), dtw.warping_path(a, b, **kwargs)

    @staticmethod
    def _path_lag(path: list) -> float:
        arr = np.asarray(path, dtype=float)
        return float(np.mean(arr[:, 1] - arr[:, 0]))

    def identify_lead_lag(self) -> dict:
        cols = list(self.data.columns)
        n = len(cols)
        D = np.zeros((n, n))
        L = np.zeros((n, n))
        for i in range(n):
            for j in range(i + 1, n):
                d, path = self.compute_dtw(self.data[cols[i]], self.data[cols[j]])
                lag = self._path_lag(path)
                D[i, j] = D[j, i] = d
                L[i, j], L[j, i] = lag, -lag
        self.distance_matrix_ = pd.DataFrame(D, index=cols, columns=cols)
        self.lag_matrix_ = pd.DataFrame(L, index=cols, columns=cols)
        return {"distance": self.distance_matrix_, "lag": self.lag_matrix_}

    def forecast(self, leader: str, follower: str, lag: Optional[int] = None) -> pd.Series:
        if lag is None:
            if self.lag_matrix_ is None:
                self.identify_lead_lag()
            lag = int(round(self.lag_matrix_.loc[leader, follower]))
        return self.data[leader].shift(max(int(lag), 0)).rename(f"{follower}_hat")

In [ ]:
ll = LeadLagDTW(prices, sakoe_chiba_radius=20)
res = ll.identify_lead_lag()

print("Matrice de distance DTW :")
print(res["distance"].round(2))
print("\nMatrice lead-lag (ligne mene colonne si > 0) :")
print(res["lag"].round(2))

res["distance"].to_csv(os.path.join(OUT, "04_dtw_distance.csv"))
res["lag"].to_csv(os.path.join(OUT, "04_lag_matrix.csv"))

## Task 5 - Tuning et Validation

Optimisation Optuna de la bande Sakoe-Chiba avec validation croisee
temporelle (TimeSeriesSplit, fenetre expansive). Metriques : MSE, MAE,
RMSE et distance de Wasserstein (sensible aux queues de distribution).

Le forecast naif "shift" sert de baseline ; les RMSE absolus sont eleves
car on predit le prix de Silver avec celui de Gold decale sans
recalibration de niveau. La valeur du test est comparative.

In [ ]:
@dataclass
class ModelTuningValidation:
    data: pd.DataFrame
    leader: str
    follower: str
    n_splits: int = 5
    best_params_: dict = field(default_factory=dict, init=False)

    @staticmethod
    def _metrics(y_true: pd.Series, y_pred: pd.Series) -> dict:
        from scipy.stats import wasserstein_distance

        df = pd.concat([y_true, y_pred], axis=1).dropna()
        df.columns = ["y", "yhat"]
        if df.empty:
            return {"mse": np.nan, "mae": np.nan, "rmse": np.nan, "wass": np.nan}
        err = df["y"] - df["yhat"]
        mse = float((err ** 2).mean())
        return {
            "mse": mse, "mae": float(err.abs().mean()), "rmse": float(np.sqrt(mse)),
            "wass": float(wasserstein_distance(df["y"].values, df["yhat"].values)),
        }

    def _cv_score(self, radius: int) -> float:
        from sklearn.model_selection import TimeSeriesSplit

        tscv = TimeSeriesSplit(n_splits=self.n_splits)
        scores = []
        for train_idx, test_idx in tscv.split(np.arange(len(self.data))):
            train, test = self.data.iloc[train_idx], self.data.iloc[test_idx]
            ll_local = LeadLagDTW(train[[self.leader, self.follower]], sakoe_chiba_radius=radius)
            ll_local.identify_lead_lag()
            lag = max(int(round(ll_local.lag_matrix_.loc[self.leader, self.follower])), 0)
            yhat = test[self.leader].shift(lag)
            scores.append(self._metrics(test[self.follower], yhat)["rmse"])
        return float(np.nanmean(scores))

    def tune_model(self, n_trials: int = 20, radius_range=(2, 30)) -> dict:
        import optuna

        optuna.logging.set_verbosity(optuna.logging.WARNING)
        study = optuna.create_study(direction="minimize")
        study.optimize(
            lambda t: self._cv_score(t.suggest_int("sakoe_chiba_radius", *radius_range)),
            n_trials=n_trials, show_progress_bar=False,
        )
        self.best_params_ = study.best_params
        return {"best_params": study.best_params, "best_value": study.best_value, "study": study}

    def validate_model(self, radius: Optional[int] = None) -> dict:
        r = radius if radius is not None else self.best_params_.get("sakoe_chiba_radius", 10)
        ll_local = LeadLagDTW(self.data[[self.leader, self.follower]], sakoe_chiba_radius=r)
        ll_local.identify_lead_lag()
        lag = max(int(round(ll_local.lag_matrix_.loc[self.leader, self.follower])), 0)
        yhat = self.data[self.leader].shift(lag)
        return {"lag": lag, "radius": r, **self._metrics(self.data[self.follower], yhat)}

In [ ]:
cols = list(prices.columns)
leader, follower = cols[0], cols[1]
print(f"Hypothese de travail : {leader} mene {follower}")

mt = ModelTuningValidation(prices, leader=leader, follower=follower, n_splits=5)
tune = mt.tune_model(n_trials=10)
print(f"Meilleurs parametres : {tune['best_params']}   RMSE CV : {tune['best_value']:.4f}")
print(f"Validation full-sample : {mt.validate_model()}")

## Task 6 - Entropy Transfer Graph

Pipeline mathematique :

1. `S = exp(-lambda * D)` : similarites a partir des distances DTW
2. `P_i = S_i / sum(S_ij)` : distribution par ligne
3. `H(i) = -sum p log p` : entropie de Shannon par actif
4. `M_ij = 1 - JSD(p_i, p_j)` : matrice de transfert (JSD symetrique, bornee)
5. `Delta = 1 - M`, embedding MDS 2D
6. Graphe networkx : noeuds = actifs, aretes ponderees par M

Une entropie faible = profil de similarite concentre = leader marque.

In [ ]:
def _shannon(p: np.ndarray) -> float:
    p = p[p > 0]
    return float(-(p * np.log(p)).sum())


def _jsd(p: np.ndarray, q: np.ndarray) -> float:
    m = 0.5 * (p + q)
    def kl(a, b):
        mask = (a > 0) & (b > 0)
        return float((a[mask] * np.log(a[mask] / b[mask])).sum())
    return 0.5 * kl(p, m) + 0.5 * kl(q, m)


@dataclass
class EntropyTransferGraph:
    distance_matrix: pd.DataFrame
    lam: Optional[float] = None
    similarity_: Optional[pd.DataFrame] = field(default=None, init=False)
    transfer_matrix_: Optional[pd.DataFrame] = field(default=None, init=False)
    entropy_: Optional[pd.Series] = field(default=None, init=False)
    embedding_: Optional[pd.DataFrame] = field(default=None, init=False)

    def compute_embeddings(self, method: str = "mds", n_components: int = 2) -> dict:
        D = self.distance_matrix.values.astype(float)
        cols = list(self.distance_matrix.columns)

        if self.lam is None:
            offdiag = D[~np.eye(len(D), dtype=bool)]
            med = np.median(offdiag) if offdiag.size else 1.0
            self.lam = float(np.log(2) / med) if med > 0 else 1.0

        S = np.exp(-self.lam * D)
        np.fill_diagonal(S, 1.0)
        self.similarity_ = pd.DataFrame(S, index=cols, columns=cols)
        P = S / S.sum(axis=1, keepdims=True)
        self.entropy_ = pd.Series([_shannon(P[i]) for i in range(len(cols))], index=cols, name="H")

        n = len(cols)
        M = np.ones((n, n))
        for i in range(n):
            for j in range(i + 1, n):
                M[i, j] = M[j, i] = 1.0 - _jsd(P[i], P[j])
        self.transfer_matrix_ = pd.DataFrame(M, index=cols, columns=cols)

        Delta = np.clip(1.0 - M, 0.0, None)
        np.fill_diagonal(Delta, 0.0)

        if method == "mds":
            from sklearn.manifold import MDS
            emb = MDS(n_components=n_components, dissimilarity="precomputed",
                      normalized_stress="auto", random_state=0).fit_transform(Delta)
        elif method == "spectral":
            from sklearn.manifold import SpectralEmbedding
            emb = SpectralEmbedding(n_components=n_components, affinity="precomputed",
                                    random_state=0).fit_transform(M)
        else:
            raise ValueError(f"method inconnue : {method}")

        self.embedding_ = pd.DataFrame(emb, index=cols, columns=[f"dim{i+1}" for i in range(n_components)])
        return {"similarity": self.similarity_, "transfer": self.transfer_matrix_,
                "entropy": self.entropy_, "embedding": self.embedding_, "lambda": self.lam}

    def plot_graph(self, threshold: float = 0.5, ax=None):
        import networkx as nx

        if self.transfer_matrix_ is None or self.embedding_ is None:
            self.compute_embeddings()
        M = self.transfer_matrix_
        G = nx.Graph()
        for c in M.columns:
            G.add_node(c)
        for i, a in enumerate(M.columns):
            for b in M.columns[i + 1:]:
                w = float(M.loc[a, b])
                if w >= threshold:
                    G.add_edge(a, b, weight=w)
        try:
            cent = nx.eigenvector_centrality_numpy(G, weight="weight")
        except Exception:
            cent = dict(G.degree(weight="weight"))
        pos = {c: self.embedding_.loc[c, ["dim1", "dim2"]].values for c in M.columns}
        if ax is None:
            _, ax = plt.subplots(figsize=(8, 6))
        sizes = [800 + 2000 * cent.get(c, 0) for c in G.nodes]
        weights = [G[u][v]["weight"] for u, v in G.edges]
        nx.draw_networkx_nodes(G, pos, node_size=sizes, node_color="#4a90e2", alpha=0.9, ax=ax)
        nx.draw_networkx_edges(G, pos, width=[3 * w for w in weights], alpha=0.5, ax=ax)
        nx.draw_networkx_labels(G, pos, font_size=10, ax=ax)
        ax.set_title("Entropy Transfer Graph")
        ax.set_axis_off()
        return ax, G

In [ ]:
etg = EntropyTransferGraph(res["distance"])
emb = etg.compute_embeddings(method="mds")
print(f"lambda auto : {emb['lambda']:.4f}")
print("Entropies de Shannon (faible = leader marque) :")
print(emb["entropy"].sort_values().round(3))

emb["transfer"].to_csv(os.path.join(OUT, "06_transfer_matrix.csv"))
emb["entropy"].to_csv(os.path.join(OUT, "06_entropy.csv"))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
etg.plot_graph(threshold=float(np.median(emb["transfer"].values)), ax=ax)
fig.savefig(os.path.join(OUT, "06_entropy_graph.png"), dpi=120, bbox_inches="tight")
plt.show()

## Task 7 - Synthese

Lecture finale du graphe : l'actif d'entropie minimale est le **leader
structurel** (profil de similarite tres concentre), celui d'entropie
maximale est le **plus reactif** (consommateur d'information).

Implications pratiques :

- Trading : un leader identifie peut servir de filtre directionnel sur
  ses suiveurs (cf. matrice lead-lag pour les delais).
- Risk management : les noeuds centraux (faible entropie) propagent les
  chocs, a surveiller en priorite.
- Limites : DTW est symetrique par construction, la directionalite vient
  uniquement du warping path. Pour une vraie causalite, implementer la
  Transfer Entropy de Schreiber.

In [ ]:
ent = emb["entropy"].sort_values()
print(f"Leader le plus fort (entropie min) : {ent.index[0]}  H = {ent.iloc[0]:.3f}")
print(f"Plus reactif      (entropie max) : {ent.index[-1]}  H = {ent.iloc[-1]:.3f}")
print(f"\nTous les outputs ont ete ecrits dans : {OUT}/")